# Minh Tuan: M5
EMA10, EMA20, EMA60, MACD5, 20, 5
# Nen dong nam tren EMA10 va Hist > 0 va MACD > Signal: Buy
# Nen dong nam duoi EMA20: Thi thuc hien lenh Buy (Co la mua)
# Nen dong nam duoi EMA60: Thi thuc hien lenh Sell All (Close All)

# Buoc 1

In [57]:
# Import duoc Common 
import sys 
sys.path.append('../Common')
import CommonMT5
import MetaTrader5 as mt5
from datetime import datetime, timedelta

symbol = 'BTCUSD'
from_date = (datetime.now() - timedelta(days=5)).strftime('%Y-%m-%d')
to_date = (datetime.now() + timedelta(days=2)).strftime('%Y-%m-%d')

timeframe = mt5.TIMEFRAME_M5
data = CommonMT5.CommonMT5.loaddataMT5_FromTo(symbol, from_date, to_date, timeframe)
data

,Datetime,Open,High,Low,Close,Volume
0,2025-09-29 00:05:00,110856.5,110906.0,110841.0,110899.0,198
1,2025-09-29 00:10:00,110899.0,110933.5,110882.0,110912.0,260
2,2025-09-29 00:15:00,110901.0,110909.0,110749.5,110793.0,294
3,2025-09-29 00:20:00,110793.5,110845.0,110793.5,110811.0,179
4,2025-09-29 00:25:00,110815.0,110824.0,110755.0,110755.0,192
...,...,...,...,...,...,...
1359,2025-10-03 17:40:00,120783.0,120894.5,120701.0,120773.0,390
1360,2025-10-03 17:45:00,120765.0,121066.0,120765.0,120952.0,401
1361,2025-10-03 17:50:00,120951.5,121126.0,120901.0,121088.0,359
1362,2025-10-03 17:55:00,121088.0,121106.0,120965.0,121041.0,284


# Buoc 2

In [ ]:
# Nen dong nam tren EMA10 > EMA20 > EMA60 va Hist > 0 va MACD > Signal: Buy
# Nen dong nam duoi EMA20: Thi thuc hien lenh Buy (Co la mua)

import talib

data['EMA10'] = talib.EMA(data['Close'], timeperiod=10)
data['EMA20'] = talib.EMA(data['Close'], timeperiod=20)
data['EMA60'] = talib.EMA(data['Close'], timeperiod=60)

macd, signal, hist = talib.MACD(
    data['Close'].values, fastperiod=5, slowperiod=20, signalperiod=5
)

data['MACD'] = macd
data['Signal'] = signal
data['Hist'] = hist  # or: macd - signal

data['DieuKien1'] = (data['Close'] > data['EMA10']) & (data['Close'] > data['EMA20']) & (data['Close'] > data['EMA60']) & (data['Hist'] > 0) & (data['MACD'] > data['Signal'])
data['DieuKien2'] = (data['Close'] < data['EMA20']) & (data['Close'] > data['EMA60'])

# DieuKien1 buy lan 1, roi sau do xet DieuKien2: DieuKien1 Buy thi khong vao lenh (dieu kien de xet thoi) va DieuKien2 < EMA20 va DieuKien2 > EMA60 thi Buy.
# Diều kiện 2 xẩy ra khi truóc đó price có lên trên MA10 xong cắt xuống dưới MA20 thì mới tính

data

,Datetime,Open,High,Low,Close,Volume,EMA10,EMA20,EMA60,MACD,Signal,Hist,DieuKien1,DieuKien2,Stop_Loss
0,2025-09-29 00:05:00,110856.5,110906.0,110841.0,110899.0,198,NaN,NaN,NaN,NaN,NaN,NaN,False,False,NaN
1,2025-09-29 00:10:00,110899.0,110933.5,110882.0,110912.0,260,NaN,NaN,NaN,NaN,NaN,NaN,False,False,NaN
2,2025-09-29 00:15:00,110901.0,110909.0,110749.5,110793.0,294,NaN,NaN,NaN,NaN,NaN,NaN,False,False,110749.5
3,2025-09-29 00:20:00,110793.5,110845.0,110793.5,110811.0,179,NaN,NaN,NaN,NaN,NaN,NaN,False,False,110749.5
4,2025-09-29 00:25:00,110815.0,110824.0,110755.0,110755.0,192,NaN,NaN,NaN,NaN,NaN,NaN,False,False,110749.5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1359,2025-10-03 17:40:00,120783.0,120894.5,120701.0,120773.0,390,120550.931022,120502.571853,120412.968464,129.161406,51.191058,77.970348,True,False,120401.0
1360,2025-10-03 17:45:00,120765.0,121066.0,120765.0,120952.0,401,120623.852654,120545.374533,120430.641629,193.114306,98.498807,94.615498,True,False,120499.0
1361,2025-10-03 17:50:00,120951.5,121126.0,120901.0,121088.0,359,120708.243081,120597.053149,120452.194362,257.939410,151.645675,106.293735,True,False,120701.0
1362,2025-10-03 17:55:00,121088.0,121106.0,120965.0,121041.0,284,120768.744339,120639.333802,120471.499465,277.661238,193.650863,84.010375,True,False,120765.0


In [59]:
# Yêu cầu đủ 3 nến mới có giá trị
data['Stop_Loss'] = data['Low'].rolling(window=3, min_periods=3).min()
data

,Datetime,Open,High,Low,Close,Volume,EMA10,EMA20,EMA60,MACD,Signal,Hist,DieuKien1,DieuKien2,Stop_Loss
0,2025-09-29 00:05:00,110856.5,110906.0,110841.0,110899.0,198,NaN,NaN,NaN,NaN,NaN,NaN,False,False,NaN
1,2025-09-29 00:10:00,110899.0,110933.5,110882.0,110912.0,260,NaN,NaN,NaN,NaN,NaN,NaN,False,False,NaN
2,2025-09-29 00:15:00,110901.0,110909.0,110749.5,110793.0,294,NaN,NaN,NaN,NaN,NaN,NaN,False,False,110749.5
3,2025-09-29 00:20:00,110793.5,110845.0,110793.5,110811.0,179,NaN,NaN,NaN,NaN,NaN,NaN,False,False,110749.5
4,2025-09-29 00:25:00,110815.0,110824.0,110755.0,110755.0,192,NaN,NaN,NaN,NaN,NaN,NaN,False,False,110749.5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1359,2025-10-03 17:40:00,120783.0,120894.5,120701.0,120773.0,390,120550.931022,120502.571853,120412.968464,129.161406,51.191058,77.970348,True,False,120401.0
1360,2025-10-03 17:45:00,120765.0,121066.0,120765.0,120952.0,401,120623.852654,120545.374533,120430.641629,193.114306,98.498807,94.615498,True,False,120499.0
1361,2025-10-03 17:50:00,120951.5,121126.0,120901.0,121088.0,359,120708.243081,120597.053149,120452.194362,257.939410,151.645675,106.293735,True,False,120701.0
1362,2025-10-03 17:55:00,121088.0,121106.0,120965.0,121041.0,284,120768.744339,120639.333802,120471.499465,277.661238,193.650863,84.010375,True,False,120765.0
